In [3]:
%pip install pytube youtube-transcript-api langchain-community langchain langchain-openai yt-dlp --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [4]:
import re
from pytube import YouTube
from langchain_core.tools import tool
from IPython.display import display, JSON
import yt_dlp
from typing import List, Dict
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
import json

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Suppress pytube errors
import logging
pytube_logger = logging.getLogger('pytube')
pytube_logger.setLevel(logging.ERROR)

# Suppress yt-dlp warnings
yt_dpl_logger = logging.getLogger('yt_dlp')
yt_dpl_logger.setLevel(logging.ERROR)

In [8]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

import os

llm = ChatOpenAI(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    model="Qwen/Qwen3.5-397B-A17B:novita"
)

##### Defining a Video ID Extraction Tool

Define `extract_video_id` as a tool to pull the 11-character YouTube video ID from a URL, since many YouTube API actions (like transcript retrieval) require the ID, not the full link. It uses regex to support standard, shortened, and embedded URL formats.

In [9]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.
    
    Args:
        url (str): A YouTube URL containing a video ID.

    Returns:
        str: Extracted video ID or error message if parsing fails.
    """
    
    # Regex pattern to match video IDs
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else "Error: Invalid YouTube URL"

##### Testing the Video ID Extraction Tool

Test extract_video_id to confirm it’s registered in LangChain by printing its name (LLM reference), description (usage guidance), and bound function reference (what gets called).

In [10]:
print(extract_video_id.name)
print("----------------------------")
print(extract_video_id.description)
print("----------------------------")
print(extract_video_id.func)

extract_video_id
----------------------------
Extracts the 11-character YouTube video ID from a URL.

Args:
    url (str): A YouTube URL containing a video ID.

Returns:
    str: Extracted video ID or error message if parsing fails.
----------------------------
<function extract_video_id at 0x11db87c40>


##### Testing Tool Execution

Test `extract_video_id` by running it on a real YouTube URL with `.run()` to verify direct execution and output.

In [12]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")
extract_video_id

StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x11db87c40>)

### Tool List

Create a tools list to organize all `@tool-decorated` tool objects in one place; it doesn’t execute anything or set call order, it just lets you pass all tools at once via `llm.bind_tools(tools)`. Add `extract_video_id` to this list so the LLM can use it when needed.

In [13]:
tools = []
tools.append(extract_video_id)

##### Defining a Transcript Fetching Tool

Create a tool that uses `YouTubeTranscriptApi` to fetch a video transcript from a video ID (plus optional language), then joins transcript segments into one text string or returns an error message if retrieval fails.

In [14]:
from youtube_transcript_api import YouTubeTranscriptApi


@tool
def fetch_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetches the transcript of a YouTube video.
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """
    
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id, languages=[language])
        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

Let's test the fetch_transcript tool by directly calling it with the `.run()` method on a specific video ID. This will attempt to retrieve the transcript for the video with ID `"hfIUstzHs9A"` in the default English language.

In [15]:
fetch_transcript.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

Adding the `fetch_transcript` tool to your tools list.

In [16]:
tools.append(fetch_transcript)

##### Defining a YouTube Search Tool

Create a tool that uses PyTube’s Search class to find videos by query and return matches as dictionaries with title, video ID, and short URL. This helps discover relevant videos when no specific URL is known.

In [17]:
from pytube import Search
from langchain.tools import tool
from typing import List, Dict

@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.
    
    Args:
        query (str): The search term to look for on YouTube
        
    Returns:
        List of dictionaries containing video titles and IDs in format:
        [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
        Returns error message if search fails
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

Now, you'll test your `search_youtube` tool by calling it with the `.run()` method and the search query "Generative AI." This will return a list of YouTube videos related to generative AI.

In [18]:
search_out=search_youtube.run("Generative AI")
display(JSON(search_out))

<IPython.core.display.JSON object>

Appending the `search_youtube` tool to tools list.

In [19]:
tools.append(search_youtube)

##### Defining a Metadata Extraction Tool

Create a tool that uses `yt-dlp` to extract detailed YouTube video metadata from a URL, such as title, views, duration, channel, likes, comments, and chapters.

In [20]:
@tool
def get_full_metadata(url: str) -> dict:
    """Extract metadata given a YouTube URL, including title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters', [])
        }

Now, you'll test your `get_full_metadata` tool by running it on a specific YouTube video URL. This will extract comprehensive information about the video with ID "qWHaMrR5WHQ" without downloading the actual video content.

**Note: If you find any issues with the given video link below, try any Youtube video link of your choosing.**

In [21]:
meta_data=get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
display(JSON(meta_data))

<IPython.core.display.JSON object>

Adding the `get_full_metadata` tool to your tools list.

In [22]:
tools.append(get_full_metadata)

##### Defining a Thumbnail Retrieval Tool

Create a tool that uses `yt-dlp` to fetch all available YouTube video thumbnails, returning each thumbnail’s URL, width, height, and formatted resolution.

In [23]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """
    Get available thumbnails for a YouTube video using its URL.
    
    Args:
        url (str): YouTube video URL (any format)
        
    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order
    """
    
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
            info = ydl.extract_info(url, download=False)
            
            thumbnails = []
            for t in info.get('thumbnails', []):
                if 'url' in t:
                    thumbnails.append({
                        "url": t['url'],
                        "width": t.get('width'),
                        "height": t.get('height'),
                        "resolution": f"{t.get('width', '')}x{t.get('height', '')}".strip('x')
                    })
            
            return thumbnails

    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

Now, you'll test your `get_thumbnails` tool by running it on a specific YouTube video URL. This will extract information about all available thumbnail images for the video.

In [24]:
thumbnails=get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")

display(JSON(thumbnails))

<IPython.core.display.JSON object>

Now, let's add the `get_thumbnails` tool to your tools list.

In [25]:
tools.append(get_thumbnails)

###  Binding tools

Now bind your tool collection to the LLM so it can call your custom YouTube tools during conversations when needed to answer user requests.


In [26]:
llm_with_tools = llm.bind_tools(tools)

`bind_tools()` sends each tool’s name, description, and parameter schema to the LLM in a standard format so it can decide when and how to call the right tool for a user request.

In [27]:
for tool in tools:
    schema = {
   "name": tool.name,
   "description": tool.description,
   "parameters": tool.args_schema.schema() if tool.args_schema else {},
   "return": tool.return_type if hasattr(tool, "return_type") else None}
    display(JSON(schema))
    

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

##### How the LLM Calls a Tool

Define a sample query asking for a YouTube video summary to show how the LLM interprets natural language and selects the right bound tools to fulfill the request.

In [28]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english


In [29]:
messages = [HumanMessage(content = query)]
print(messages)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]


##### LangChain Tool Binding Process

Invoke the LLM with your YouTube summary query and store the result in `response_1`, which includes both reply text and any structured tool calls (tool names + parameters) the model chooses.

In [30]:
response_1 = llm_with_tools.invoke(messages)
response_1

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_92877a77fff042b3b8e0190a', 'function': {'arguments': '{"url": "https://www.youtube.com/watch?v=T-D1OfcDW1M"}', 'name': 'extract_video_id'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 784, 'total_tokens': 948, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 119, 'rejected_prediction_tokens': 0, 'text_tokens': 164, 'image_tokens': 0, 'video_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'text_tokens': 784, 'image_tokens': 0, 'video_tokens': 0}}, 'model_name': 'qwen/qwen3.5-397b-a17b', 'system_fingerprint': '', 'id': '5352c4cdfdce75e6497e3783a89916e9', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019cc69e-272c-71a1-9674-7d3bd79dce0b-0', tool_calls=[{'nam

In [31]:
messages.append(response_1)

##### Extracting Tool Call Information

Use `tool_calls_1 = response_1.tool_calls` to get the structured tool calls (selected tool + parameters) for execution.

Create a name-to-tool dictionary so you can look up and invoke tools programmatically from tool-call names returned by the LLM.

In [32]:
tool_mapping = {
    "get_thumbnails" : get_thumbnails,
    "extract_video_id": extract_video_id,
    "fetch_transcript": fetch_transcript,
    "search_youtube": search_youtube,
    "get_full_metadata": get_full_metadata
}

In [33]:
tool_calls_1 = response_1.tool_calls
display(JSON(tool_calls_1))

<IPython.core.display.JSON object>

In [36]:
tool_name=tool_calls_1[0]['name']
print(tool_name)

tool_call_id =tool_calls_1[0]['id']
print(tool_call_id)

args=tool_calls_1[0]['args']
print(args)

extract_video_id
call_92877a77fff042b3b8e0190a
{'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}


In [38]:
my_tool=tool_mapping[tool_calls_1[0]['name']]
video_id =my_tool.invoke(tool_calls_1[0]['args'])
video_id

'T-D1OfcDW1M'

In [39]:
messages.append(ToolMessage(content = video_id, tool_call_id = tool_calls_1[0]['id']))

In [40]:
response_2 = llm_with_tools.invoke(messages)
response_2

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_bbadddb6510345209481adce', 'function': {'arguments': '{"video_id": "T-D1OfcDW1M", "language": "en"}', 'name': 'fetch_transcript'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 850, 'total_tokens': 947, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 45, 'rejected_prediction_tokens': 0, 'text_tokens': 97, 'image_tokens': 0, 'video_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'text_tokens': 850, 'image_tokens': 0, 'video_tokens': 0}}, 'model_name': 'qwen/qwen3.5-397b-a17b', 'system_fingerprint': '', 'id': '1cacbcbc283490097b26b6b41acc4d80', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019cc6a2-e889-7f10-9d11-a768467a2d94-0', tool_calls=[{'name': 'fetch_t

In [41]:
messages.append(response_2)

In [42]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[{'name': 'fetch_transcript',
  'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'},
  'id': 'call_bbadddb6510345209481adce',
  'type': 'tool_call'}]

In [43]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! 

In [44]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

In [45]:
summary = llm_with_tools.invoke(messages)
summary

AIMessage(content='## Summary: Retrieval-Augmented Generation (RAG)\n\n**Speaker:** Marina Danilevsky, Senior Research Scientist at IBM Research\n\n### Main Topic\nThis video explains **Retrieval-Augmented Generation (RAG)**, a framework designed to make large language models (LLMs) more accurate and up-to-date.\n\n### Key Problems with LLMs\nThe speaker identifies two major challenges with standard large language models:\n\n1. **No Source Citation** - LLMs give answers without providing evidence or sources\n2. **Outdated Information** - LLMs rely on training data that becomes stale over time\n\n### The RAG Solution\nInstead of relying solely on what the LLM learned during training, RAG adds a **retrieval step**:\n\n1. User asks a question\n2. LLM first retrieves relevant content from a data store (internet, documents, policies, etc.)\n3. LLM combines retrieved content with the user\'s question\n4. LLM generates an answer with evidence\n\n### Benefits of RAG\n- ✅ **Up-to-date informati

### Automating the Tool-Calling Process

Instead of manually repeating detect-call, extract args, run tool, and return result, automate it with a function that reads each tool call, finds the function via `tool_mapping`, executes it with the provided arguments, and returns a `ToolMessage` with the same `tool_call_id` so the LLM can correctly match responses to requests, especially across multi-step or parallel calls.

In [46]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id=tool_call["id"]
        )

        

##### Building the Summarization Chain

Build `summarization_chain` with `|` to run steps in sequence: convert prompt to `HumanMessage`, call tool-enabled LLM, extract and execute tool calls, append tool results to message history, repeat as needed, then use `RunnableLambda` to return only the final summary text.

In [47]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

summarization_chain = (
    # Start with initial query
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLM call (extract video ID)
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process first tool call
    | RunnablePassthrough.assign(
        tool_messages=lambda x: [
            execute_tool(tc) for tc in x["ai_response"].tool_calls
        ]
    )
    # Update message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call (fetch transcript)
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [
            execute_tool(tc) for tc in x["ai_response2"].tool_calls
        ]
    )
    # Final message update
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Generate final summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Return just the summary text
    | RunnableLambda(lambda x: x["summary"])
)



In [48]:
# Usage
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

Video Summary:
 This video provides an overview of **LangChain**, an open-source orchestration framework designed to simplify the development of applications powered by Large Language Models (LLMs).

Here is a summary of the key points covered:

*   **What is LangChain?** It is a framework (available in Python and JavaScript) that acts as a generic interface for almost any LLM. It allows developers to build complex applications by chaining together different components, minimizing the amount of code needed.
*   **Core Components (Abstractions):**
    *   **LLMs:** Provides a standard interface to use various models (e.g., GPT-4, Llama 2) simply by providing an API key.
    *   **Prompts:** Uses templates to formalize instructions, examples (few-shot prompting), and output formats without hardcoding.
    *   **Chains:** The core workflow mechanism that links multiple functions sequentially (e.g., retrieving data -> summarizing it -> answering a question).
    *   **Indexes:** Tools for 

##### Creating the Initial Message Setup

Use `RunnablePassthrough.assign` to transform the input `query` into an initial message list containing one `HumanMessage`.

In [49]:
initial_setup = RunnablePassthrough.assign(
    messages=lambda x: [HumanMessage(content=x["query"])]
)

##### Defining the First LLM Interaction

Create the second chain step to send formatted messages to the tool-enabled LLM and store its output as `ai_response`.

In [50]:
first_llm_call = RunnablePassthrough.assign(
    ai_response=lambda x: llm_with_tools.invoke(x["messages"])
)

##### Processing the First Tool Call

Execute each LLM-requested tool via `execute_tool`, append the LLM response and `ToolMessage` results to message history, and pass the updated state to the next LLM step.

In [51]:
first_tool_processing = RunnablePassthrough.assign(
    tool_messages=lambda x: [
        execute_tool(tc) for tc in x["ai_response"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
)

##### Defining the Second LLM Interaction

Create the next chain step to send the updated message history (including first tool results) back to the LLM.

In [52]:
second_llm_call = RunnablePassthrough.assign(
    ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
)

##### Processing the Second Tool Call

This step executes the LLM’s second tool call (often transcript retrieval), creates `ToolMessage` results, and updates message history for the final summarization step.

In [53]:
second_tool_processing = RunnablePassthrough.assign(
    tool_messages2=lambda x: [
        execute_tool(tc) for tc in x["ai_response2"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
)

##### Generating the Final Summary

Use the full message history (query, tool calls, and tool results), invoke the LLM one final time, and return only the response content via `RunnableLambda` as the summary text.

In [54]:
final_summary = RunnablePassthrough.assign(
    summary=lambda x: llm_with_tools.invoke(x["messages"]).content
) | RunnableLambda(lambda x: x["summary"])

##### Assembling the complete chain

In [55]:
chain = (
    initial_setup
    | first_llm_call
    | first_tool_processing
    | second_llm_call
    | second_tool_processing
    | final_summary
)

In [56]:
query = {"query": "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"}
result = summarization_chain.invoke(query)
print("Video Summary:\n", result)

Video Summary:
 


Test with a different query

In [57]:
query = {"query": "Get top 3 youtube videos in India and their metadata"}
try:
    result = summarization_chain.invoke(query)
    print("Video Summary:\n", result)
except Exception as e:
    print("Non-critical network error:", e)

result

Video Summary:
 


''

### Recursive Chain Flow

The current chain is fixed to two tool calls, but real use cases may need a variable number; next, build a recursive chain that keeps calling tools dynamically until enough information is gathered.

In [58]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.messages import HumanMessage, ToolMessage
import json

def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        content = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
    except Exception as e:
        content = f"Error: {str(e)}"
    
    return ToolMessage(
        content=content,
        tool_call_id=tool_call["id"]
    )

#### Defining the core processing logic

1. Identifies the most recent message in the conversation
2. Extracts all tool calls from that message and executes them in parallel using your `execute_tool` helper
3. Updates the message history by adding the tool response messages
4. Gets the next response from the language model based on the updated conversation
5. Returns the complete updated message history with both tool responses and the new LLM response

In [59]:
def process_tool_calls(messages):
    """Recursive tool call processor"""
    last_message = messages[-1]
    
    # Execute all tool calls in parallel
    tool_messages = [
        execute_tool(tc) 
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    
    # Add tool responses to message history
    updated_messages = messages + tool_messages
    
    # Get next LLM response
    next_ai_response = llm_with_tools.invoke(updated_messages)
    
    return updated_messages + [next_ai_response]

##### Creating the recursive stopping condition

This function checks the last message for tool calls (safely via `getattr`) and returns `True` to continue recursion or `False` when the LLM gives a final answer with no more tool requests.


In [60]:
def should_continue(messages):
    """Check if you need another iteration"""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))

##### Implementing the Recursive Function

This function drives dynamic tool calling by checking `should_continue`, processing tool calls and recursing with updated messages when needed, or returning final message history when no more tool calls remain; then wrap it with `RunnableLambda` for LangChain compatibility.

In [61]:
def _recursive_chain(messages):
    """Recursively process tool calls until completion"""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages

recursive_chain = RunnableLambda(_recursive_chain)

##### Building the Complete Universal Chain

Assemble a three-step universal chain: convert query to `HumanMessage`, get the first response from the tool-enabled LLM, then hand off to the recursive chain to process as many tool calls as needed until a final answer is produced.

In [62]:
universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages + [llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [63]:
query_us = {"query": "Show top 3 US trending videos with metadata and thumbnails"}

try:
    response = universal_chain.invoke(query_us)
    print("\nUS Trending Videos:\n", response[-1])
except Exception as e:
    print("Non-critical network error while fetching US trending videos:", e)


US Trending Videos:
 content='I don\'t have access to a tool that can fetch YouTube\'s trending videos. The available tools I have are:\n\n- Search YouTube for videos with specific queries\n- Extract video IDs from URLs\n- Get metadata for specific video URLs\n- Get thumbnails for specific video URLs\n- Fetch transcripts for specific videos\n\nHowever, I can help you in a couple of ways:\n\n1. **Search for popular/trending topics**: If you tell me what topics are currently trending (like "US news", "viral videos", "trending music", etc.), I can search for those and get metadata and thumbnails for the results.\n\n2. **Analyze specific videos**: If you have specific YouTube URLs of trending videos you\'d like me to analyze, I can get their full metadata and thumbnails.\n\nWould you like me to search for videos on a specific trending topic, or do you have specific video URLs you\'d like me to analyze?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tok

### 📝 Exercise 1: Try a different video with a Youtube link


In [67]:
# TODO
video_id = "sQSQBYHR0ms"

Video Summary:
 Excellent summary! You've captured the core arguments and technical details very well. The video raises important questions about the realistic role of quantum computing in AI development.

### Additional Insights from the Video:

**Key Question Raised:**
- Are we aiming quantum computing at the right target? The video suggests that trying to make quantum computers compete directly with GPU parallel computing for AI training might be counterintuitive.

**Interesting Analogy:**
- Just as we still use CPUs for everyday tasks (browsing, Excel, emails) while GPUs handle heavy parallel computation, quantum computers might serve different specialized purposes rather than replacing GPUs for LLM training.

**Future Possibilities:**
- The video hints that quantum computing might enable entirely new ways of processing knowledge that classical computers couldn't consider, rather than just speeding up existing AI training methods.

**Industry Reality:**
- Multiple companies (IonQ, 

### 📝 Exercise 2: Extract the video ID


In [69]:
# TODO
youtube_url = "https://www.youtube.com/watch?v=sQSQBYHR0ms"
video_id = extract_video_id(youtube_url)
video_id


'sQSQBYHR0ms'

### 📝 Exercise 3: Collect all necessary data about the video in one go


In [70]:
# TODO
video_metadata = get_full_metadata.run(youtube_url)
video_metadata


{'title': 'Quantum Computing and AI',
 'views': 138311,
 'duration': 640,
 'channel': 'Caleb Writes Code',
 'likes': 3964,
 'comments': 118,
 'chapters': [{'start_time': 0.0, 'title': 'Intro', 'end_time': 20.0},
  {'start_time': 20.0, 'title': 'LLM Training', 'end_time': 155.0},
  {'start_time': 155.0, 'title': 'CPU to GPU to QPU', 'end_time': 374.0},
  {'start_time': 374.0,
   'title': 'Quantum Computing Industry',
   'end_time': 478.0},
  {'start_time': 478.0, 'title': 'Scaling in Quantum', 'end_time': 567.0},
  {'start_time': 567.0, 'title': 'Quantum Computer Now', 'end_time': 640}]}

### 📝 Exercise 4: Get video transcript


In [71]:
transcript = fetch_transcript.run(video_id)
transcript

"One of the biggest expectations around quantum computers is that one day quantum computers will train AI models significantly faster. And that begs the question, what is it about quantum computers that actually make this faster? Let's start with what goes into training a large language model and look at how quantum computers might disrupt the AI industry. Quick shout out to Sammy for helping me research this video. Let's start with an open model from Meta called Llama 3.1 from 2024. Lama 3.1 is a 405 billion parameter model that used 15 trillion tokens for pre-training, which means using just one GPU, it would take around 4,486 years just to train this model. And in order to reduce this insane number to something more tangible, we can simply throw more GPUs at the problem. Let's say 16,000 GPUs all dedicated to training this model, which brings the number down to 3 months as opposed to 4,000 years. In classical computers, we call this type of work parallel computing. And parallel comp

### 📝 Exercise 5: Get video thumbnails


In [72]:
thumbnails = get_thumbnails.run(youtube_url)
thumbnails

[{'url': 'https://i.ytimg.com/vi/sQSQBYHR0ms/3.jpg',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi_webp/sQSQBYHR0ms/3.webp',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi/sQSQBYHR0ms/2.jpg',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi_webp/sQSQBYHR0ms/2.webp',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi/sQSQBYHR0ms/1.jpg',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi_webp/sQSQBYHR0ms/1.webp',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi/sQSQBYHR0ms/mq3.jpg',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi_webp/sQSQBYHR0ms/mq3.webp',
  'width': None,
  'height': None,
  'resolution': ''},
 {'url': 'https://i.ytimg.com/vi/sQSQBYHR0ms/mq2.jpg',
  'width': None,
  'height': None,
  'resolut

### Let's have a comprehensive prompt to be passed to LLM to generate a summary

In [73]:
prompt = f"""
Please analyze this YouTube video and provide a comprehensive summary.

VIDEO TITLE: {video_metadata['title']}
CHANNEL: {video_metadata['channel']}
VIEWS: {video_metadata['views']}
DURATION: {video_metadata['duration']} seconds
LIKES: {video_metadata['likes']}

TRANSCRIPT EXCERPT:
{transcript[:3000]}... (transcript truncated for brevity)

Based on this information, please provide:
1. A concise summary of the video content (3-5 bullet points)
2. The main topics or themes discussed
3. The intended audience for this content
4. A brief analysis of why this video might be performing well (or not)
"""

### 📝 Exercise 6: Single LLM invocation with all the data


In [75]:
messages = [HumanMessage(content=prompt)]
response = llm.invoke(messages)

###  📝 Exercise 7: Display the comprehensive analysis


In [78]:
print(response.content)

Based on the video metadata and transcript excerpt provided, here is the comprehensive analysis:

### 1. Concise Summary
*   **Investigates Quantum AI Hype:** The video explores the common expectation that quantum computers will eventually train AI models significantly faster than classical systems.
*   **Benchmarking with Llama 3.1:** It uses Meta's Llama 3.1 (405 billion parameters) as a case study to illustrate the massive computational scale required, noting it would take one GPU over 4,000 years to train versus 3 months using 16,000 GPUs.
*   **Explains Classical Parallelism:** The content details how current AI training relies on classical parallel computing to divide massive workloads across GPU clusters to manage approximately $10^{25}$ floating-point operations (FLOPs).
*   **Questions Quantum Mechanics:** It challenges the misconception that quantum computers speed up processes by simply doing all math simultaneously, setting up a comparison between quantum capabilities and e